In [ ]:
import gymnasium as gym
import numpy as np
import cv2
import tensorflow as tf
from keras.models import load_model

# 1. Environment aur Model Load Karein
env = gym.make("CartPole-v1" , render_mode="rgb_array")
q_net = load_model("sarsa_cartpole_model.h5")


In [2]:

# 2. Policy Function (Evaluation ke liye explore=0.0 use hoga)
def policy(state, explore=0.0):
    if tf.random.uniform(shape=(), maxval=1.0) <= explore:
        return np.random.randint(0, 2)
    
    q_values = q_net(state)
    return int(tf.argmax(q_values[0]).numpy())


In [3]:

# 3. Evaluation Loop
for episode in range(5):
    done = False
    
    # FIX 1: Safely unpack reset and reshape Initial State
    state, info = env.reset()
    state = tf.expand_dims(tf.convert_to_tensor(state, dtype=tf.float32), 0)
    
    total_reward = 0
    
    while not done:
        frame = env.render()
        
        # FIX 2: Convert RGB to BGR for OpenCV
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        cv2.imshow('CartPole Evaluation', frame_bgr)
        cv2.waitKey(50)  # 50ms thoda smooth view dega
        
        # Action with explore=0.0 (Greedy/Optimal moves only)
        action = policy(state, explore=0.0)
        
        # FIX 3: Safely unpack step (5 values)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Update state properly for the next iteration
        state = tf.expand_dims(tf.convert_to_tensor(next_state, dtype=tf.float32), 0)
        
        total_reward += reward
        
    print(f"Evaluation Episode {episode + 1} | Total Reward: {total_reward}")
        
env.close()
cv2.destroyAllWindows()

Evaluation Episode 1 | Total Reward: 9.0
Evaluation Episode 2 | Total Reward: 10.0
Evaluation Episode 3 | Total Reward: 10.0
Evaluation Episode 4 | Total Reward: 11.0
Evaluation Episode 5 | Total Reward: 9.0
